# Lexington, Missouri — Land Value Tax Model

**City of Lexington, Lafayette County, Missouri** — 4:1 split-rate LVT model, city general-revenue levy only, preserving all exemptions.

Parcel source: **Regrid** ("Data with Purpose", CLE licensed), Lafayette County geoparquet refreshed 2025-04-15, **clipped to the City of Lexington municipal limits** (Census place **2941870**, "Lexington city", 3,346 land acres). The pre-clipped input is `data/regrid_lexington_cityclip.parquet` (gitignored — licensed data, never committed); the clip boundary is `data/city_boundary_2941870.geojson`.

**Boundary change (2026-07-27 re-run).** The prior run used the postal `scity == 'LEXINGTON'` filter, which pulled in 1,102 rural parcels lying outside the municipal limits that do not pay the city levy. Greg approved re-running against the true city-limits clip. This notebook now loads the pre-clipped city-limits file directly.

Follows the LVT Shift canonical 7-section notebook template.

## Section 1 — Imports and Setup

In [ ]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'lexington_mo'
STATE_FIPS = '29'              # Missouri
COUNTY_FIPS = '107'            # Lafayette County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

# Parcel-map identity columns (Regrid schema)
PARCEL_ID_COL = 'parcelnumb'
OWNER_NAME_COL = 'owner'
OWNER_ADDRESS_COL = 'address'
PARCEL_URL_TEMPLATE = None     # Lafayette County has no stable public per-parcel deep link

# Missouri statutory assessment ratios (RSMo 137.115) applied to market value
RATIO_RESIDENTIAL = 0.19
RATIO_COMMERCIAL = 0.32
RATIO_AGRICULTURAL = 0.12

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load / Build Parcel Data

The staged input is the Lafayette County Regrid file **already clipped to the City of Lexington municipal limits** (Census place 2941870): `data/regrid_lexington_cityclip.parquet`, 2,521 rows / 2,467 unique parcels. This replaces the prior run's county-wide load + postal `scity == 'LEXINGTON'` filter, which had over-included 1,102 rural parcels outside the city.

We:
1. load the pre-clipped city-limits file (no `scity` filter needed — the clip is the municipal boundary),
2. **dedupe stacked geometries**: some parcels appear as several rows sharing `parcelnumb` with different geometry fragments but **identical** `landval`/`improvval`/`parval` (the value is a parcel attribute repeated on each fragment — summing would triple-count). We keep one record per `parcelnumb`, unioning the geometry fragments,
3. cache the result to `data/parcels.gpq` (the canonical per-parcel cache that downstream stages read). The cache is rebuilt unconditionally on this run so the stale postal-run cache is overwritten.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
CITYCLIP_PATH = DATA_DIR / 'regrid_lexington_cityclip.parquet'

# Pre-clipped City-of-Lexington universe (Census place 2941870). Rebuild the
# canonical parcels.gpq cache unconditionally so any stale postal-run cache is replaced.
lex = gpd.read_parquet(CITYCLIP_PATH).rename_geometry('geometry')
print(f"City-clip raw rows (pre-dedupe): {len(lex):,}  unique parcelnumb: {lex['parcelnumb'].nunique():,}")

# Dedupe stacked duplicate geometries on parcelnumb; union geometry fragments.
dup_ids = lex['parcelnumb'][lex['parcelnumb'].duplicated(keep=False)].unique()
singles = lex[~lex['parcelnumb'].isin(dup_ids)].copy()
dgroups = lex[lex['parcelnumb'].isin(dup_ids)].copy()
dissolved = dgroups.dissolve(by='parcelnumb', aggfunc='first', as_index=False)
gdf = pd.concat([singles, dissolved], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=lex.crs).copy()

gdf.to_parquet(PARCEL_PATH)
print(f"Deduped to {len(gdf):,} unique parcels ({len(dup_ids)} had stacked geometry); cached to {PARCEL_PATH}")

# Numeric value fields
for c in ['landval', 'improvval', 'parval', 'agval', 'll_gisacre']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0.0)
print("geometry valid:", gdf.geometry.is_valid.mean(), " CRS:", gdf.crs)
print("built (improvval>0): {:,}   vacant (improvval==0): {:,}".format(
    int((gdf['improvval'] > 0).sum()), int((gdf['improvval'] <= 0).sum())))
print("market value sums — land ${:,.0f}  improvement ${:,.0f}  parcel ${:,.0f}".format(
    gdf['landval'].sum(), gdf['improvval'].sum(), gdf['parval'].sum()))

## Section 3 — Classify and Validate

**Exemptions (preserved).** Regrid carries no assessor exempt flag for this county (`taxamt`, `homestead_exemption`, `owntype` are all null on the Lexington subset). We infer the Missouri constitutional exemptions (Art. X §6 — government, religious, educational/charitable) from `lbcs_ownership_desc`. These parcels are held out of the taxable base and the reform, exactly as the assessor's certified valuation excludes them.

**Assessment class → Missouri ratio.** Regrid values are `parvaltype == 'MARKET'`, i.e. market value, **not** Missouri assessed value. Missouri assesses by subclass (RSMo 137.115): residential 19 %, commercial 32 %, agricultural 12 %. We assign each taxable parcel a class from the strongest available signals (`lbcs_ownership_desc`, `zoning`/`zoning_type`, `lbcs_structure_desc`, presence of a building):

- **Agricultural (12 %)** — AG-zoned, unimproved, > 5 acres (genuine farm/open land; excludes AG-zoned house lots).
- **Commercial (32 %)** — business-entity ownership, or commercial/industrial zoning.
- **Residential (19 %)** — everything else (dwellings, in-town house and vacant lots).

`usecode`/`usedesc` are entirely null on this subset, so the class comes from the LBCS + zoning signals above rather than an assessor use code.

In [ ]:
own = gdf['lbcs_ownership_desc']
zon = gdf['zoning']
ztype = gdf['zoning_type']
struct = gdf['lbcs_structure_desc']
has_bldg = gdf['improvval'] > 0

EXEMPT_OWNERS = ['City, Village, Township, etc.', 'State government', 'Federal government',
                 'County, Parish, Province, etc.', 'Nonprofit religious', 'Nonprofit educational']
gdf['full_exmp'] = own.isin(EXEMPT_OWNERS).astype(int)
exempt = gdf['full_exmp'] == 1

is_ag = (~exempt) & (zon == 'AG') & (~has_bldg) & (gdf['ll_gisacre'] > 5)
is_comm = (~exempt) & (~is_ag) & ((own == 'Businesses and commercial entities') | ztype.isin(['Commercial', 'Industrial']))
is_res = (~exempt) & (~is_ag) & (~is_comm)

gdf['assess_ratio'] = np.select([exempt, is_ag, is_comm, is_res],
                                 [0.0, RATIO_AGRICULTURAL, RATIO_COMMERCIAL, RATIO_RESIDENTIAL],
                                 default=RATIO_RESIDENTIAL)
gdf['assess_class'] = np.select([exempt, is_ag, is_comm, is_res],
                                 ['exempt', 'agricultural', 'commercial', 'residential'], default='residential')
print(gdf['assess_class'].value_counts())

# Property category for the report
cat = pd.Series('Other', index=gdf.index)
cat[is_res] = 'Single Family Residential'
cat[is_comm] = 'Commercial'
cat[is_ag] = 'Agricultural'
cat[exempt] = 'Exempt'
gdf['PROPERTY_CATEGORY'] = cat
# Mandatory override: $0 improvement -> Vacant Land (catches empty lots incl. unimproved ag land)
gdf.loc[(gdf['improvval'] <= 0) & (~exempt), 'PROPERTY_CATEGORY'] = 'Vacant Land'
print(gdf.loc[~exempt, 'PROPERTY_CATEGORY'].value_counts())

## Section 4 — Current Tax Model + Revenue Gate

**Levy.** City of Lexington general-revenue municipal real-property levy: **$0.5481 per $100** of assessed value = **5.481 mills** (per $1,000). Source: Missouri State Auditor, *2025 Property Tax Rates*, Report No. 2026-006 (Jan 2026), City of Lexington / General Revenue — certified assessed valuation **$54,948,146** × $0.5481/$100 = **$301,171** levied.

**Market → assessed + calibration to the certified control total.** We convert Regrid market values to Missouri assessed values with the per-subclass ratios above. On the **city-limits clip** the resulting real-property taxable base totals **~$35.1M** — about **36% below** the auditor's certified $54,948,146, and materially *further* below it than the postal run's ~$49.2M (K was 1.116 there). We apply a **single uniform calibration factor** `K = certified_AV / Σ(market × ratio)` so the taxable base equals the certified valuation; this run's `K ≈ 1.57`.

**Why the gap widened when we clipped to the city (this is expected, not a regression).** The certified $54,948,146 is the aggregate general-revenue valuation, which in Missouri includes **personal property** (vehicles, business equipment — assessed at 33⅓%) and state-assessed utilities that **no real-property parcel file can contain**; personal property is routinely ~30–45% of a small Missouri city's roll. The prior postal run's extra 1,102 rural parcels had coincidentally padded the real-property base up toward $49.2M, masking that gap and making K look small (1.116). Removing them — which is *correct*, since those parcels lie outside the municipal limits and pay no city levy — exposes the true city-limits real-property base (~$35.1M) and the real+personal split embedded in the certified figure. The larger K is therefore a symptom of the certified control total not being a pure real-property number, **not** of missing city parcels.

**K is a level-only adjustment and cancels from the reform.** Because `K` is uniform across land and improvements, it **cancels out of the revenue-neutral split-rate solution**: the land/improvement millages and every per-parcel *percentage* change are identical with or without it. It affects only the *absolute* dollar figures, which it anchors to the certified levy. Caveat: anchoring the full $301,171 (real+personal) levy onto the real-property base means reported *absolute* dollars implicitly carry the personal-property share of the levy; the distributional results (who wins/loses, and by what %) are unaffected. See Limitations.

In [ ]:
RATE_PER_1000 = 5.481          # mills (per $1,000 assessed)
CERTIFIED_AV = 54_948_146      # MO State Auditor 2026-006, City of Lexington General Revenue
OFFICIAL_REVENUE = 301_171     # certified AV x rate (real+personal aggregate; see limitations)

# Assessed (market x subclass ratio) before calibration
gdf['assessed_land_raw'] = gdf['landval'].clip(lower=0) * gdf['assess_ratio']
gdf['assessed_improvement_raw'] = gdf['improvval'].clip(lower=0) * gdf['assess_ratio']

raw_base = (gdf.loc[~exempt, 'assessed_land_raw'] + gdf.loc[~exempt, 'assessed_improvement_raw']).sum()
K = CERTIFIED_AV / raw_base
print(f"Raw taxable assessed base (Regrid x ratios): ${raw_base:,.0f}")
print(f"Certified assessed valuation (auditor):      ${CERTIFIED_AV:,.0f}")
print(f"Uniform calibration factor K = {K:.4f}")

gdf['taxable_land_value'] = gdf['assessed_land_raw'] * K
gdf['taxable_improvement_value'] = gdf['assessed_improvement_raw'] * K
gdf['taxable_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']
gdf['millage_rate'] = RATE_PER_1000

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"\nModeled current revenue: ${current_revenue:,.0f}   Official: ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"
print("Revenue gate: PASS")

## Section 5 — Split-Rate Model (4:1)

Land taxed at 4× the improvement rate, revenue-neutral to the city general-revenue levy. Fully-exempt parcels are held out of the solver and dropped from the modeled output (they carry no signal), and reported by count only.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}   (current flat: {RATE_PER_1000:.4f})")
print(f"Revenue check: ${new_revenue:,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

### Artifact scan (Gate 5)

Read the category table against economic priors before trusting it.

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3))

**Gate 5 reading.** Vacant Land shows the large increase expected of underused land; built categories (SFR, Commercial) show modest decreases with plausible ~0.8 building shares — no ceiling clustering, no distinct categories pinned at one extreme value, no $0-building placeholder cluster. Land is a low share of total value (~16-18%), which is economically real for a small rural county-seat where structures dominate parcel value; it is not an artifact. Result is believable.

## Section 6 — Exploration Chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % Change')
ax.axvline(0, color='k', lw=0.8)
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census Join + Export (canonical closing pattern)

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

## Limitations & Data Notes

- **Parcel source:** Regrid (licensed "Data with Purpose"), Lafayette County, refreshed 2025-04-15, **clipped to the City of Lexington municipal limits** (Census place 2941870, "Lexington city"). Input `data/regrid_lexington_cityclip.parquet`; boundary `data/city_boundary_2941870.geojson`. Gitignored — never committed.
- **Boundary basis (changed 2026-07-27):** prior run filtered the county file by postal `scity == 'LEXINGTON'` (3,569 unique parcels), which over-included **1,102 rural parcels** outside the municipal limits that pay no city levy. This run uses the true city-limits clip: **2,521 raw rows → 2,467 unique parcels** (47 stacked-geometry ids unioned).
- **Market → assessed:** Regrid values are `MARKET`. Converted to Missouri assessed value via statutory subclass ratios (residential 19 %, commercial 32 %, agricultural 12 %; RSMo 137.115). Subclass inferred from `lbcs_ownership_desc` / `zoning` / `lbcs_structure_desc` because `usecode`/`usedesc` are null on this subset.
- **Calibration (K ≈ 1.57, up from 1.12):** the city-limits real-property assessed base (~$35.1M) is ~36 % below the county-certified assessed valuation ($54,948,146), so the uniform calibration factor rose from ~1.12 (postal run) to ~1.57. The gap widened *because* the clip is now correct: the certified figure is a **real + personal + state-assessed aggregate**, and Missouri personal property (vehicles/equipment at 33⅓ %) is typically ~30–45 % of a small city's roll — value no real-property parcel file can contain. The postal run's rural parcels had coincidentally padded the base and hidden this. K is uniform, so it is **level-only**: it does not affect the split-rate millages or any per-parcel percentage change, only the absolute dollar figures (which it anchors to the certified levy).
- **Revenue figure caveat:** the auditor's $301,171 is the aggregate general-revenue levy (real + personal). This model covers **real property only**; the calibrated base is anchored to the certified assessed valuation so modeled current revenue reconciles to the levy. Because the personal-property share of the levy is anchored onto real property via K, reported *absolute* dollars run high relative to a real-property-only levy; distributional (%) outcomes are unaffected.
- **Exemptions:** no assessor exempt flag in the data; government / religious / educational parcels inferred from ownership type and held out (preserving existing exemptions).
- **Agricultural land:** Missouri assesses farmland on *productive* value, which the market-based Regrid values only approximate. On the city clip only 2 genuine unimproved farm parcels remain (the rural AG parcels that dominated this class in the postal run are now correctly excluded); any residual unimproved parcels fold into the "Vacant Land" report category by the standard $0-improvement override.